# Scaleout

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import awkward as ak
from hist import Hist
from coffea import processor
from coffea.nanoevents import PHYSLITESchema
from coffea.analysis_tools import PackedSelection

PHYSLITESchema.warn_missing_crossrefs = False

warnings.filterwarnings(
    "ignore",
    message="Skipping ",
    category=UserWarning,
)

In [ ]:
from importlib.metadata import version

for package in ["numpy", "awkward", "uproot", "coffea"]:
    print(f"# {package}: v{version(package)}")

In [ ]:
def filter_name(name):
    """
    Load only the properties/variables needed.
    """
    return name in (
        "EventInfoAuxDyn.mcEventWeights",
        #
        "AnalysisElectronsAuxDyn.pt",
        "AnalysisElectronsAuxDyn.eta",
        "AnalysisElectronsAuxDyn.phi",
        "AnalysisElectronsAuxDyn.m",
        "AnalysisElectronsAuxDyn.DFCommonElectronsLHLoose",
        "AnalysisElectronsAuxDyn.charge",
        #
        "AnalysisMuonsAuxDyn.pt",
        "AnalysisMuonsAuxDyn.eta",
        "AnalysisMuonsAuxDyn.phi",
        "AnalysisMuonsAuxDyn.m",
        "AnalysisMuonsAuxDyn.charge",
        "AnalysisMuonsAuxDyn.quality",
        #
        "AnalysisJetsAuxDyn.pt",
        "AnalysisJetsAuxDyn.eta",
        "AnalysisJetsAuxDyn.phi",
        "AnalysisJetsAuxDyn.m",
        #
        "BTagging_AntiKt4EMPFlowAuxDyn.DL1dv01_pb",
        "BTagging_AntiKt4EMPFlowAuxDyn.DL1dv01_pc",
        "BTagging_AntiKt4EMPFlowAuxDyn.DL1dv01_pu",
    )

In [ ]:
GeV = 1000


def object_selection(events):
    """
    Select objects based on kinematic and quality criteria
    """

    electrons = events.Electrons
    muons = events.Muons

    electron_reqs = (
        (electrons.pt / GeV > 20)
        & (np.abs(electrons.eta) < 2.47)
        & (electrons.DFCommonElectronsLHLoose == 1)
    )

    muon_reqs = (muons.pt / GeV > 20) & (np.abs(muons.eta) < 2.7) & (muons.quality == 2)

    # only keep objects that pass our requirements
    electrons = electrons[electron_reqs]
    muons = muons[muon_reqs]

    return electrons, muons


def region_selection(electrons, muons):
    """
    Select events based on object multiplicity
    """

    selections = PackedSelection(dtype="uint64")
    # basic selection criteria
    selections.add("exactly_4e", ak.num(electrons) == 4)
    selections.add("total_e_charge_zero", ak.sum(electrons.charge, axis=1) == 0)
    selections.add("exactly_0m", ak.num(muons) == 0)
    # selection criteria combination
    selections.add(
        "4e0m", selections.all("exactly_4e", "total_e_charge_zero", "exactly_0m")
    )

    return selections.all("4e0m")


def calculate_inv_mass(electrons):
    """
    Construct invariant mass observable
    """

    # reconstruct Higgs as 4e system
    candidates = ak.combinations(electrons, 4)
    e1, e2, e3, e4 = ak.unzip(candidates)
    candidates["p4"] = e1 + e2 + e3 + e4
    higgs_mass = candidates["p4"].mass
    observable = ak.flatten(higgs_mass / GeV)

    return observable

In [ ]:
# create histogram with observables
class create_histograms(processor.ProcessorABC):
    def process(self, events):
        hist_4e0m = (
            Hist.new.Reg(50, 100, 150, name="m_inv", label=r"$m_{inv.}(4e)$ [GeV]")
            .StrCat([], name="process", label="Process", growth=True)
            .Weight()
        )
    
        # read metadata
        dataset = events.metadata["dataset"]
        process_name = events.metadata["process"]
        x_sec = events.metadata["xsec"]
        gen_filt_eff = events.metadata["genFiltEff"]
        k_factor = events.metadata["kFactor"]
        sum_of_weights = events.metadata["sumOfWeights"]
    
        # as mentined already, the actual analysis code remains the same!
        # select objects and events
        el, mu = object_selection(events)
        selection_4e0m = region_selection(el, mu)
    
        # normalization for MC
        lumi = 36100.0  # /pb This is the luminosity (the amount of real data collected) corresponding to the open data released
        xsec_weight = x_sec * gen_filt_eff * k_factor * lumi / sum_of_weights
        print(f"Processing {process_name} with xsec weight {xsec_weight}")
        mc_weight = events.EventInfo[selection_4e0m]["mcEventWeights"][:, 1]
    
        # observable calculation and histogram filling
        inv_mass = calculate_inv_mass(el[selection_4e0m])
        hist_4e0m.fill(inv_mass, weight=mc_weight * xsec_weight, process=process_name)
    
        return {dataset: hist_4e0m}

    def postprocess(self, accumulator):
        pass

In [ ]:
xcache_caching_server = "root://xcache.af.uchicago.edu:1094//"

The metadata for open data is available by the [metadata table](https://opendata.atlas.cern/docs/documentation/overview_data/data_research_2024#metadata).

In [ ]:
fileset = {
    "Higgs": {
        "files": {
            f"{xcache_caching_server}root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000001.pool.root.1": "CollectionTree",
            f"{xcache_caching_server}root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000002.pool.root.1": "CollectionTree",
            f"{xcache_caching_server}root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000005.pool.root.1": "CollectionTree",
            f"{xcache_caching_server}root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000006.pool.root.1": "CollectionTree",
            f"{xcache_caching_server}root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000007.pool.root.1": "CollectionTree",
            f"{xcache_caching_server}root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000008.pool.root.1": "CollectionTree",
            f"{xcache_caching_server}root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000009.pool.root.1": "CollectionTree",
            f"{xcache_caching_server}root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000010.pool.root.1": "CollectionTree",
            f"{xcache_caching_server}root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000011.pool.root.1": "CollectionTree",
            f"{xcache_caching_server}root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000012.pool.root.1": "CollectionTree",
            f"{xcache_caching_server}root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000013.pool.root.1": "CollectionTree",
            f"{xcache_caching_server}root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000014.pool.root.1": "CollectionTree",
            f"{xcache_caching_server}root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000016.pool.root.1": "CollectionTree",
            f"{xcache_caching_server}root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000017.pool.root.1": "CollectionTree",
            f"{xcache_caching_server}root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000018.pool.root.1": "CollectionTree",
            f"{xcache_caching_server}root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000019.pool.root.1": "CollectionTree",
            f"{xcache_caching_server}root://eospublic.cern.ch//eos/opendata/atlas/rucio/mc20_13TeV/DAOD_PHYSLITE.38191712._000020.pool.root.1": "CollectionTree",
        },
        "metadata": {
            "process": "Higgs",
            "xsec": 28.3,
            "genFiltEff": 1.240e-04,
            "kFactor": 1.45,
            "sumOfWeights": 114108.08,
        },
    }
}

In [ ]:
# set up runner

run = processor.Runner(
    executor = processor.FuturesExecutor(workers=4, compression=None),
    schema=PHYSLITESchema,
    savemetrics=True,
)

In [ ]:
%%time

# pre-process
chunk_generator = run.preprocess(fileset)

In [ ]:
%%time

# execute
out, metrics = run(
    chunk_generator,
    processor_instance=create_histograms(),
)
out, metrics

In [ ]:
# stack all the histograms together, as we processed each file separately
full_histogram = sum(hist for hist in out.values())

In [ ]:
plot_dir = Path().cwd() / "plots"
plot_dir.mkdir(exist_ok=True)

In [ ]:
# plot
artists = full_histogram.plot(histtype="fill")

ax = artists[0].stairs.axes
ax.legend()
ax.set_ylabel("A.U.")

fig = ax.get_figure()
fig.savefig(plot_dir / "higgs_mass.png")

## Scaleout and the virtual-array backend

Older versions of these tutorials scaled out by building a **dask-awkward task graph** and calling `.compute()`. Coffea 2026.7 changes the execution model: `NanoEvents` are **virtual arrays**, so each branch stays on disk and is materialized on demand. A single `Runner` streams through the data chunk-by-chunk with no lazy graph and no `.compute()` — the `FuturesExecutor` run above already worked this way.

Two ways to scale the *same* processor:

* **Multiple local cores** — swap `IterativeExecutor` → `FuturesExecutor` (as above); it uses the standard-library `concurrent.futures`.
* **A distributed cluster** — use `processor.DaskExecutor` with a `dask.distributed.Client`, shown next. This distributes *chunks* to dask workers and processes each eagerly on virtual arrays; it is **not** the dask-awkward DAG / `.compute()` path, just a different place to run the same per-chunk work.

The larger refactor unifying all of these behind one interface (`coffea.compute`) is previewed at the end of the notebook.

### Scaling out to a cluster with `DaskExecutor`

`DaskExecutor` hands each preprocessed chunk to a `dask.distributed` worker and gathers the filled histograms back — a map-reduce over chunks, with the virtual-array reads happening on the workers. On an Analysis Facility you connect to the cluster shown in the JupyterLab Dask extension; on a laptop, `Client()` spins up a local cluster so the example still runs. The processor and fileset are unchanged from the `FuturesExecutor` run above — only the executor differs.

In [ ]:
from dask.distributed import Client

# On an Analysis Facility, point this at the cluster scheduler shown in the
# JupyterLab Dask extension (drag it in, or connect explicitly), e.g.:
#     client = Client("tls://localhost:8786")
# On a laptop, a bare Client() starts a local cluster so the example still runs.
client = Client()
client

In [ ]:
%%time

# Same processor, same fileset -- only the executor changes. DaskExecutor
# distributes chunks to the dask workers; each chunk is read as virtual arrays
# and processed eagerly (no dask-awkward graph, no .compute()).
run = processor.Runner(
    executor=processor.DaskExecutor(client=client, compression=None),
    schema=PHYSLITESchema,
    savemetrics=True,
)

out, metrics = run(
    run.preprocess(fileset),
    processor_instance=create_histograms(),
)
out, metrics

In [ ]:
full_histogram = sum(hist for hist in out.values())

artists = full_histogram.plot(histtype="fill")
ax = artists[0].stairs.axes
ax.legend()
ax.set_ylabel("A.U.")
ax.get_figure().savefig(plot_dir / "higgs_mass_dask.png")

## Preview: upcoming coffea features

The cells below preview features that are **not yet released** — they live in open *draft* pull requests against the coffea repository. Each demo is **guarded**: it detects whether the feature is present (by import / signature introspection, never by version number, since these are unreleased branches) and prints a gentle note instead of raising if it is missing. This section is therefore safe to "Run All" in the default environment.

To actually exercise the demos, launch one of the preview environments defined in `pixi.toml`:

```bash
pixi run -e preview jupyter lab           # pydantic dataset-tools extensions: PRs #1579, #1600, #1601
pixi run -e preview-compute jupyter lab    # the coffea.compute execution refactor: PR #1470
```

Those environments install coffea straight from the PR branches, so the exact API may drift before release.

In [ ]:
# --- Feature detection for the preview demos below ---------------------------
# These upcoming features live in *unreleased* draft PRs, so we never rely on a
# version number; each is detected by import / signature introspection. When a
# feature is absent the demo cells print a gentle note instead of raising, so
# this whole section is safe to "Run All" in the default environment.
import importlib.util
import inspect
from pathlib import Path

import coffea
from coffea.dataset_tools import preprocess as _preprocess


def _has_param(func, name):
    try:
        return name in inspect.signature(func).parameters
    except (TypeError, ValueError):
        return False


HAS_PP_BACKENDS = _has_param(_preprocess, "backend")             # draft PR #1579
HAS_PP_METADATA = _has_param(_preprocess, "metadata_extractor")  # draft PR #1600
HAS_MUTABLE_STEPS = importlib.util.find_spec("coffea.dataset_tools.mutable_steps") is not None  # draft PR #1601
HAS_COMPUTE = importlib.util.find_spec("coffea.compute") is not None  # draft PR #1470


def preview_note(feature, pr):
    print(
        f"[preview] '{feature}' is not available in this coffea build ({coffea.__version__}).\n"
        f"          It ships in draft PR {pr}. To try it, launch a preview environment:\n"
        f"            pixi run -e preview jupyter lab           # pydantic dataset-tools extensions (#1579/#1600/#1601)\n"
        f"            pixi run -e preview-compute jupyter lab    # coffea.compute execution refactor (#1470)"
    )


# A small, network-free sample so the preview demos run wherever this repo is
# checked out (they fall back gracefully if it is missing).
_preview_file = Path("../columnar/data/SMHiggsToZZTo4L.root")
_preview_fileset = {
    "demo": {"files": {str(_preview_file): "Events"}, "metadata": {"xsec": 1.0}}
}

print(f"coffea {coffea.__version__}")
for _flag in ["HAS_PP_BACKENDS", "HAS_PP_METADATA", "HAS_MUTABLE_STEPS", "HAS_COMPUTE"]:
    print(f"  {_flag} = {globals()[_flag]}")

### 1. Pydantic dataset specifications

*Released in coffea 2026.7 (PR #1528) — the foundation the previews build on.*

Filesets can now be expressed as validated `pydantic` models (`DataGroupSpec` / `DatasetSpec` / `ROOTFileSpec`, ...). Malformed filesets fail fast with clear errors, and the models carry form and metadata around for the tools below.

In [ ]:
# 1. Pydantic dataset specifications (released in coffea 2026.7, PR #1528)
# The classic "dict-in / dict-out" fileset still works, but datasets can now be
# expressed as *validated* pydantic models, catching malformed filesets early.
from coffea.dataset_tools import ModelFactory, DatasetSpec

spec = ModelFactory.dict_to_datasetspec(_preview_fileset["demo"])
print("type:", type(spec).__name__, "| is DatasetSpec:", isinstance(spec, DatasetSpec))
print("validated metadata:", dict(spec.metadata))
print("file specs:", [type(fs).__name__ for fs in spec.files.values()])
# round-trip back to a plain dict when a legacy API needs one
_roundtrip = ModelFactory.datasetspec_to_dict(spec)

### 2. Non-dask preprocessing backends — draft PR #1579

Released `preprocess()` builds a **dask-awkward** graph to discover file chunks. PR #1579 adds a `backend=` switch (`"iterative"`, `"futures"`, `"dask"`) so preprocessing can run with **no dask dependency**, plus a dedicated `preprocess_rntuple()` for RNTuple inputs. The backend classes (`IterativeBackend`, `FuturesBackend`, ...) are explicitly designed to plug into the `coffea.compute` refactor below.

In [ ]:
# 2. Non-dask preprocessing backends  (draft PR #1579)
from coffea.dataset_tools import preprocess

if HAS_PP_BACKENDS and _preview_file.exists():
    available, report = preprocess(
        _preview_fileset,
        step_size=50_000,
        save_form=False,
        backend="iterative",  # or "futures"; "dask" reproduces the legacy path
        skip_bad_files=True,
    )
    finfo = list(available["demo"]["files"].values())[0]
    print("preprocessed with the dask-free 'iterative' backend")
    print("  steps discovered:", finfo["steps"])
    print("  num_entries:", finfo["num_entries"])
elif not HAS_PP_BACKENDS:
    preview_note("preprocess(backend=...)", "#1579")
else:
    print("[preview] sample file not found; skipping the live run.")

### 3. User-supplied metadata extraction — draft PR #1600

Computing per-dataset quantities such as the sum of generator weights normally means an extra pass over the files. PR #1600 adds `metadata_extractor` (called once per file on the open handle) and `metadata_reducer` (called once per dataset) hooks to `preprocess()`, folding that work into the preprocessing pass.

In [ ]:
# 3. User-supplied metadata extraction during preprocessing  (draft PR #1600)
from coffea.dataset_tools import preprocess

if HAS_PP_METADATA and _preview_file.exists():
    def per_file(file_handle):
        # runs once per file, on the open uproot file handle
        return {"nentries": int(file_handle["Events"].num_entries)}

    def per_dataset(per_file_meta):
        # reduce the per-file dicts into dataset-level metadata
        return {"nentries_total": sum(m["nentries"] for m in per_file_meta.values())}

    available, _ = preprocess(
        _preview_fileset,
        step_size=50_000,
        save_form=False,
        backend="iterative",
        metadata_extractor=per_file,
        metadata_reducer=per_dataset,
        skip_bad_files=True,
    )
    print("dataset metadata after extraction:", dict(available["demo"]["metadata"]))
elif not HAS_PP_METADATA:
    preview_note("preprocess(metadata_extractor=..., metadata_reducer=...)", "#1600")
else:
    print("[preview] sample file not found; skipping the live run.")

### 4. Adaptive / resizable steps — draft PR #1601

Fixed step sizes over- or under-shoot when chunk cost varies. This **prototype** adds a resizable step generator whose size can be renegotiated mid-stream through the generator `.send()` channel (the same channel `coffea.compute`'s `Computable.gen_steps` uses), plus a `run_adaptive_steps` driver governed by a `WallTimeStepPolicy`. The API is explicitly marked unstable.

In [ ]:
# 4. Adaptive / resizable steps  (draft PR #1601, prototype -- API may change)
if HAS_MUTABLE_STEPS:
    from coffea.dataset_tools.mutable_steps import resizable_steps

    gen = resizable_steps(0, 1_000, 200)
    produced = [next(gen)]
    try:
        while True:
            # after the first chunk, ask the generator to shrink the step to 100
            produced.append(gen.send(100))
    except StopIteration:
        pass
    print("resizable_steps, shrunk mid-stream via .send(100):")
    print(" ", produced)

    # Higher-level driver, operating on a preprocessed pydantic DatasetSpec:
    print(
        "\nHigher-level API (illustrative):\n"
        "    from coffea.dataset_tools.mutable_steps import (\n"
        "        iter_dataset_steps, run_adaptive_steps, WallTimeStepPolicy)\n"
        "    policy = WallTimeStepPolicy(target_seconds=30)\n"
        "    total = run_adaptive_steps(dataset_spec, work_fn, step_size=100_000, policy=policy)"
    )
else:
    preview_note("coffea.dataset_tools.mutable_steps", "#1601")

### 5. A unified execution protocol: `coffea.compute` — draft PR #1470

The largest change on the horizon. PR #1470 introduces `coffea.compute`, replacing the `Processor` / `Executor` / `Runner` trio with a single `Backend` **protocol**. Work is expressed as a `Computable` (a `Dataset` mapped through a function via `.map_steps`), handed to any backend's `.compute()`, which returns a non-blocking `Task` exposing `.result()`, `.partial_result()`, `.wait()`, and `.cancel()`. The preprocessing backends (#1579) and resizable steps (#1601) are stepping stones toward this unified interface. The one-liner it enables:

```python
with ThreadedBackend() as backend:
    total = backend.compute(dataset.map_steps(process)).result()
```

This backend is a genuine work in progress, so the demo below is guarded to show the protocol *shape* even where it does not yet fully execute.

In [ ]:
# 5. A unified execution protocol: coffea.compute  (draft PR #1470, WIP)
# PR #1470 replaces the Processor/Executor/Runner trio with a single `Backend`
# protocol. A `Computable` (a Dataset mapped through a function via .map_steps)
# is handed to any backend's .compute(), returning a non-blocking Task with
# .result()/.partial_result(). This is what tutorial scaleout could look like
# once the refactor lands:
if HAS_COMPUTE:
    from coffea.compute.data import Dataset, File, ContextDataset
    from coffea.compute.backends.threaded import ThreadedBackend

    dataset = Dataset(
        files=[File(path=str(_preview_file), steps=[(0, 50_000), (50_000, 100_000)])],
        metadata=ContextDataset(dataset_name="demo", cross_section=None),
    )

    def count(events):  # a plain callable *is* the processor
        return len(events)

    computable = dataset.map_steps(count)
    print(f"built a Computable with {len(computable)} work element(s)")
    print(
        "the target one-liner:\n"
        "    with ThreadedBackend() as backend:\n"
        "        total = backend.compute(computable).result()"
    )
    try:
        with ThreadedBackend() as backend:
            total = backend.compute(computable).result()
        print("result:", total)
    except Exception as exc:  # coffea.compute is a work-in-progress preview
        print(
            f"[preview] coffea.compute did not execute here yet ({type(exc).__name__}); "
            "the protocol shape above is the point of this WIP preview."
        )
else:
    preview_note("coffea.compute", "#1470")